In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from falsb4mpa.dataset.utils import bucket

In [2]:
used_columns = [
    'sex', #Bin
    'age_cat', #OHE
    'race', #OHE
    'juv_fel_count', #MinMax
    'juv_misd_count', #MinMax
    'juv_other_count', #MinMax
    'priors_count', #MinMax
    'c_charge_degree', #Bin
    'is_recid', #Already bin
    'is_violent_recid', #Already bin
    'two_year_recid' #Already bin
]
target = 'decile_score' # set high_risk

# Reading data

In [3]:
raw_data = pd.read_csv("../../../data/raw/compas/compas.csv", index_col=0)

In [4]:
raw_data['high_risk'] = (raw_data['decile_score'] >= 7).astype(int)

In [5]:
cols = used_columns + ['high_risk']
data = raw_data[cols]

In [6]:
data.head()

,sex,age_cat,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,two_year_recid,high_risk
id,,,,,,,,,,,,
1,Male,Greater than 45,Other,0,0,0,0,F,0,0,0,0
3,Male,25 - 45,African-American,0,0,0,0,F,1,1,1,0
4,Male,Less than 25,African-American,0,0,1,4,F,1,0,1,0
5,Male,Less than 25,African-American,0,1,0,1,F,0,0,0,1
6,Male,25 - 45,Other,0,0,0,2,F,0,0,0,0


In [7]:
print(len(data.index))

7214


In [8]:
data.isna().sum()

sex                 0
age_cat             0
race                0
juv_fel_count       0
juv_misd_count      0
juv_other_count     0
priors_count        0
c_charge_degree     0
is_recid            0
is_violent_recid    0
two_year_recid      0
high_risk           0
dtype: int64

In [9]:
data.duplicated()

id
1        False
3        False
4        False
5        False
6        False
         ...  
10996     True
10997     True
10999     True
11000     True
11001     True
Length: 7214, dtype: bool

# Binarizing

In [10]:
data['sex'].value_counts()

sex
Male      5819
Female    1395
Name: count, dtype: int64

In [11]:
data['sex'] = (data['sex'] == 'Male').astype(int)
data['sex'].value_counts()

/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14908/1893676450.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sex'] = (data['sex'] == 'Male').astype(int)


sex
1    5819
0    1395
Name: count, dtype: int64

In [12]:
data['c_charge_degree'].value_counts()

c_charge_degree
F    4666
M    2548
Name: count, dtype: int64

In [13]:
data['c_charge_degree'] = (data['c_charge_degree'] == 'M').astype(int)
data['c_charge_degree'].value_counts()

/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14908/1280120774.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['c_charge_degree'] = (data['c_charge_degree'] == 'M').astype(int)


c_charge_degree
0    4666
1    2548
Name: count, dtype: int64

# One hot enconde

In [14]:
categorical_attr = ['age_cat', 'race']

In [15]:
for attr in categorical_attr:
    data = pd.concat([data, pd.get_dummies(data[attr], prefix=attr, dtype=int)], axis=1)

In [16]:
data.head()

,sex,age_cat,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,...,high_risk,age_cat_25 - 45,age_cat_Greater than 45,age_cat_Less than 25,race_African-American,race_Asian,race_Caucasian,race_Hispanic,race_Native American,race_Other
id,,,,,,,,,,,,,,,,,,,,,
1,1,Greater than 45,Other,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1
3,1,25 - 45,African-American,0,0,0,0,0,1,1,...,0,1,0,0,1,0,0,0,0,0
4,1,Less than 25,African-American,0,0,1,4,0,1,0,...,0,0,0,1,1,0,0,0,0,0
5,1,Less than 25,African-American,0,1,0,1,0,0,0,...,1,0,0,1,1,0,0,0,0,0
6,1,25 - 45,Other,0,0,0,2,0,0,0,...,0,1,0,0,0,0,0,0,0,1


# MinMax scaler

In [17]:
continous_attr = [
    'juv_fel_count', #MinMax
    'juv_misd_count', #MinMax
    'juv_other_count', #MinMax
    'priors_count', #MinMax
]

scaler = MinMaxScaler()

In [18]:
for attr in continous_attr:
    data[attr] = scaler.fit_transform(np.array(data[attr]).reshape(-1,1))

In [19]:
data.describe()

,sex,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,two_year_recid,high_risk,age_cat_25 - 45,age_cat_Greater than 45,age_cat_Less than 25,race_African-American,race_Asian,race_Caucasian,race_Hispanic,race_Native American,race_Other
count,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000
mean,0.806626,0.003362,0.006995,0.006434,0.091379,0.353202,0.481148,0.113529,0.450652,0.276546,0.569587,0.218464,0.211949,0.512337,0.004436,0.340172,0.088301,0.002495,0.052259
std,0.394971,0.023699,0.037326,0.029505,0.128488,0.477998,0.499679,0.317261,0.497593,0.447321,0.495168,0.413233,0.408717,0.499882,0.066459,0.473800,0.283751,0.049893,0.222565
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,0.052632,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,0.000000,0.000000,0.131579,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# Reordering the columns

In [20]:
columns_order = [
    'sex', #Bin
    'race_African-American', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Native American', 'race_Other',
    'age_cat_25 - 45', 'age_cat_Greater than 45', 'age_cat_Less than 25',
    'juv_fel_count', #MinMax
    'juv_misd_count', #MinMax
    'juv_other_count', #MinMax
    'priors_count', #MinMax
    'c_charge_degree', #Bin
    'is_recid', #Already bin
    'is_violent_recid', #Already bin
    'two_year_recid', #Already bin
    'high_risk'
]

In [21]:
data.head()

,sex,age_cat,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,...,high_risk,age_cat_25 - 45,age_cat_Greater than 45,age_cat_Less than 25,race_African-American,race_Asian,race_Caucasian,race_Hispanic,race_Native American,race_Other
id,,,,,,,,,,,,,,,,,,,,,
1,1,Greater than 45,Other,0.0,0.000000,0.000000,0.000000,0,0,0,...,0,0,1,0,0,0,0,0,0,1
3,1,25 - 45,African-American,0.0,0.000000,0.000000,0.000000,0,1,1,...,0,1,0,0,1,0,0,0,0,0
4,1,Less than 25,African-American,0.0,0.000000,0.058824,0.105263,0,1,0,...,0,0,0,1,1,0,0,0,0,0
5,1,Less than 25,African-American,0.0,0.076923,0.000000,0.026316,0,0,0,...,1,0,0,1,1,0,0,0,0,0
6,1,25 - 45,Other,0.0,0.000000,0.000000,0.052632,0,0,0,...,0,1,0,0,0,0,0,0,0,1


In [22]:
data = data.drop(['race', 'age_cat'], axis=1)

In [23]:
data = data[columns_order]
data.head()

,sex,race_African-American,race_Asian,race_Caucasian,race_Hispanic,race_Native American,race_Other,age_cat_25 - 45,age_cat_Greater than 45,age_cat_Less than 25,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,two_year_recid,high_risk
id,,,,,,,,,,,,,,,,,,,
1,1,0,0,0,0,0,1,0,1,0,0.0,0.000000,0.000000,0.000000,0,0,0,0,0
3,1,1,0,0,0,0,0,1,0,0,0.0,0.000000,0.000000,0.000000,0,1,1,1,0
4,1,1,0,0,0,0,0,0,0,1,0.0,0.000000,0.058824,0.105263,0,1,0,1,0
5,1,1,0,0,0,0,0,0,0,1,0.0,0.076923,0.000000,0.026316,0,0,0,0,1
6,1,0,0,0,0,0,1,1,0,0,0.0,0.000000,0.000000,0.052632,0,0,0,0,0


# Saving data

In [24]:
print(len(data.index))

7214


In [25]:
data.to_csv('../../../data/processed/compas/compas_mpa_cat_wout_agg.csv')